# Task 2: California Housing Price Prediction (OLS vs Ridge vs Lasso)

**Goal:** predict the median house value of a California district from its census features, and compare three linear models: Ordinary Least Squares (OLS), Ridge (L2) and Lasso (L1), using RMSE, MAE and R².

**Dataset:** California Housing (20,640 districts, 8 features). The target `MedHouseVal` is in units of $100,000.

**Steps:** EDA, missing values, correlation and multicollinearity check, interaction features, scaling, hyper-parameter tuning with GridSearchCV, metrics, predicted-vs-actual and residual plots.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data

In [ ]:
df = fetch_california_housing(as_frame=True).frame
print("Shape:", df.shape)
df.head()

## 3. Exploratory data analysis

In [ ]:
print(df.describe().T.round(2))
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["MedHouseVal"], bins=50, ax=axes[0])
axes[0].set_title("Target: median house value ($100k)")
sns.scatterplot(data=df.sample(3000, random_state=RANDOM_STATE), x="Longitude", y="Latitude",
                hue="MedHouseVal", palette="viridis", s=12, ax=axes[1])
axes[1].set_title("Prices by location")
plt.tight_layout(); plt.show()
print("Rows at the price cap (5.0):", (df["MedHouseVal"] >= 5.0).sum())

**Note on missing values.** This version of the dataset has no missing values, but the pipeline below still contains a `SimpleImputer(strategy="median")` step, so the model would handle nulls if they appeared. The imputer is fitted on the training data only.

**Note on the price cap.** The target is capped at 5.0 ($500,000). Those districts will show up as a diagonal line in the residual plot later.

## 4. Correlation heatmap and multicollinearity

In [ ]:
corr = df.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

print("Correlation with the target:\n", corr["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False).round(3))

**Variance Inflation Factor (VIF)** measures multicollinearity: how well one feature can be predicted from all the others. A VIF above about 5 to 10 is a warning sign. Ridge and Lasso are designed to cope with this problem.

In [ ]:
def vif_table(frame):
    rows = []
    for col in frame.columns:
        others = frame.drop(columns=col)
        r2 = LinearRegression().fit(others, frame[col]).score(others, frame[col])
        rows.append((col, 1 / (1 - r2)))
    return pd.DataFrame(rows, columns=["feature", "VIF"]).sort_values("VIF", ascending=False).round(2)

features = df.drop(columns="MedHouseVal")
vif_table(features)

## 5. Feature engineering (interaction features)

In [ ]:
fe = df.copy()
fe["BedrmsRatio"]    = fe["AveBedrms"] / fe["AveRooms"]      # share of rooms that are bedrooms
fe["RoomsPerPerson"] = fe["AveRooms"] / fe["AveOccup"]       # space per resident
fe["Income_x_Rooms"] = fe["MedInc"] * fe["AveRooms"]         # richer areas with bigger homes
fe["Income_x_Age"]   = fe["MedInc"] * fe["HouseAge"]
fe["Lat_x_Lon"]      = fe["Latitude"] * fe["Longitude"]      # coarse location interaction

new_cols = ["BedrmsRatio", "RoomsPerPerson", "Income_x_Rooms", "Income_x_Age", "Lat_x_Lon"]
print(fe[new_cols + ["MedHouseVal"]].corr()["MedHouseVal"].drop("MedHouseVal").round(3))
fe.head()

## 6. Train/test split (80/20)

In [ ]:
X = fe.drop(columns="MedHouseVal")
y = fe["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 7. Baseline models: OLS, Ridge, Lasso

Each model sits in a `Pipeline` (impute, scale, fit). Putting the scaler inside the pipeline means it is fitted only on training data, and inside every cross-validation fold later. That avoids data leakage.

In [ ]:
def make_pipe(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler()),
                     ("model", model)])

def evaluate(name, pipe):
    pred = pipe.predict(X_test)
    return {"Model": name,
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "MAE": mean_absolute_error(y_test, pred),
            "R2": r2_score(y_test, pred)}

baseline = {"OLS": make_pipe(LinearRegression()),
            "Ridge (alpha=1)": make_pipe(Ridge(alpha=1.0)),
            "Lasso (alpha=0.01)": make_pipe(Lasso(alpha=0.01, max_iter=20000))}
for p in baseline.values():
    p.fit(X_train, y_train)

pd.DataFrame([evaluate(n, p) for n, p in baseline.items()]).round(4)

## 8. Hyper-parameter tuning with GridSearchCV (5-fold)

In [ ]:
ridge_grid = GridSearchCV(make_pipe(Ridge()),
                          {"model__alpha": np.logspace(-3, 3, 13)},
                          cv=5, scoring="neg_root_mean_squared_error")
lasso_grid = GridSearchCV(make_pipe(Lasso(max_iter=20000)),
                          {"model__alpha": np.logspace(-4, 0, 9)},
                          cv=5, scoring="neg_root_mean_squared_error")
ridge_grid.fit(X_train, y_train)
lasso_grid.fit(X_train, y_train)

print("Best Ridge alpha:", ridge_grid.best_params_["model__alpha"], " CV RMSE:", round(-ridge_grid.best_score_, 4))
print("Best Lasso alpha:", lasso_grid.best_params_["model__alpha"], " CV RMSE:", round(-lasso_grid.best_score_, 4))

In [ ]:
# CV error against alpha, to see where the sweet spot is
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, grid, title in [(axes[0], ridge_grid, "Ridge"), (axes[1], lasso_grid, "Lasso")]:
    alphas = grid.cv_results_["param_model__alpha"].data.astype(float)
    ax.semilogx(alphas, -grid.cv_results_["mean_test_score"], marker="o")
    ax.axvline(grid.best_params_["model__alpha"], color="red", ls="--", label="best alpha")
    ax.set_title(f"{title}: CV RMSE vs alpha"); ax.set_xlabel("alpha"); ax.set_ylabel("RMSE"); ax.legend()
plt.tight_layout(); plt.show()

## 9. Final comparison on the test set

In [ ]:
final = {"OLS": baseline["OLS"],
         "Ridge (tuned)": ridge_grid.best_estimator_,
         "Lasso (tuned)": lasso_grid.best_estimator_}
results = pd.DataFrame([evaluate(n, p) for n, p in final.items()]).set_index("Model")
results.round(4)

In [ ]:
results[["RMSE", "MAE"]].plot(kind="bar", figsize=(7, 4), rot=0)
plt.title("Test error (lower is better)"); plt.ylabel("Error ($100k)")
plt.show()

## 10. What did Ridge and Lasso do to the coefficients?

In [ ]:
coefs = pd.DataFrame({name: pipe.named_steps["model"].coef_ for name, pipe in final.items()},
                     index=X.columns)
print("Features Lasso set to exactly zero:", list(coefs.index[coefs["Lasso (tuned)"] == 0]))
coefs.round(3)

In [ ]:
coefs.plot(kind="barh", figsize=(9, 7))
plt.title("Standardised coefficients"); plt.axvline(0, color="black", lw=0.8)
plt.tight_layout(); plt.show()

## 11. Predicted vs actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (name, pipe) in zip(axes, final.items()):
    pred = pipe.predict(X_test)
    ax.scatter(y_test, pred, s=6, alpha=0.3)
    lim = [0, 5.5]
    ax.plot(lim, lim, color="red", lw=1.5)
    ax.set_title(f"{name}  (R2 = {r2_score(y_test, pred):.3f})")
    ax.set_xlabel("Actual ($100k)")
axes[0].set_ylabel("Predicted ($100k)")
plt.tight_layout(); plt.show()

## 12. Residual analysis

Residual = actual minus predicted. In a good model residuals are centred on zero with no pattern.

In [ ]:
best_name = results["RMSE"].idxmin()
best_pred = final[best_name].predict(X_test)
resid = y_test - best_pred
print("Best model:", best_name, "| mean residual:", round(resid.mean(), 4), "| std:", round(resid.std(), 4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(resid, bins=50, kde=True, ax=axes[0]); axes[0].set_title("Residual distribution")
axes[1].scatter(best_pred, resid, s=6, alpha=0.3)
axes[1].axhline(0, color="red"); axes[1].set_title("Residuals vs predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")
from scipy import stats
stats.probplot(resid, dist="norm", plot=axes[2]); axes[2].set_title("Q-Q plot")
plt.tight_layout(); plt.show()

## 13. Conclusions

Fill these in with the numbers from your own run:

- **Best model:** (name, RMSE, MAE, R²).
- **Ridge vs OLS:** (almost identical? why? Hint: after scaling, the data is well-conditioned and there is plenty of training data, so shrinking the weights barely helps.)
- **Lasso:** (which features, if any, were set to zero? what does that say about which features matter?)
- **Multicollinearity:** (which features had high VIF, e.g. Latitude and Longitude, AveRooms and AveBedrms?)
- **Residuals:** (centred on zero? the diagonal streak comes from the $500k price cap, and the model under-predicts the most expensive districts.)
- **Limitation:** linear models reach only about R² 0.6 here. Prices depend non-linearly on location, so a tree-based model such as Random Forest or Gradient Boosting would likely do better.